# bitladder: KWS experiments in Colab or Kaggle

Runtime: GPU (a T4 is enough). All logic lives in the repository scripts; this notebook only
clones the repo, fetches the feature cache and runs chains of experiments. Every run goes through
`run_once`: results are pushed to the private results repo after each run, finished runs are
skipped on a re-run, so after a disconnect you re-run the same cell and it continues.

**Colab**: Runtime → T4 GPU; Secrets (key icon) → `RESULTS_TOKEN` with notebook access.
**Kaggle**: File → Import notebook → GitHub URL of this file; Settings → Accelerator GPU,
Internet ON (needs a phone-verified account); Add-ons → Secrets → `RESULTS_TOKEN`.

Data options (run **one** of the two cells in section 2):

* **2a** download a prebuilt `kws_cache.tar` shared by link (no Drive mount, nothing else on
  your Drive is exposed);
* **2b** download Speech Commands v2 from tensorflow.org and rebuild the cache (~5 min).

Section 3 holds one chain per day; run the one you need. Section 4 collects the JSON files.

In [ ]:
# 1. Code (Colab or Kaggle). Every experiment below goes through run() / run_once(): RESULT and epoch lines are echoed
#    live, a crash STOPS the cell (the old `| grep ^RESULT` swallowed errors), finished runs are skipped on a re-run.
import os, sys
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
WORK = '/kaggle/working' if ON_KAGGLE else '/content'      # code + data live here; the results-repo clone goes to /tmp
REPO = 'https://github.com/Vzzzz/bitladder.git'
if not os.path.exists(f'{WORK}/bitladder'):
    !git clone -q {REPO} {WORK}/bitladder
%cd {WORK}/bitladder
!git pull -q --ff-only && git log -1 --oneline
!pip install -q -e . 2>&1 | grep -iE 'error|failed' || true
for m in [m for m in sys.modules if m == 'bitladder' or m.startswith('bitladder.')]:
    del sys.modules[m]   # a kernel that imported the old code before `git pull` would keep using it
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
from bitladder.notebook import run, run_once, done   # raises ImportError if the install above failed
run('python -m pytest -q -p no:cacheprovider', show=('=',))   # last line must say "NN passed"

In [ ]:
# 2a. Prebuilt feature cache from a link-shared file (Google Drive file id or any direct URL).
# Build it locally once: tar -cf kws_cache.tar -C data kws_cache
CACHE_FILE_ID = '1Ho0bzfCwmfxS65FYsntta4HbjkAcQSkP'   # kws_cache.tar on Google Drive (link-shared); or set CACHE_URL
CACHE_URL = ''
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/kws_cache/index.json'):
    if CACHE_FILE_ID:
        !pip install -q gdown && gdown -q {CACHE_FILE_ID} -O {WORK}/kws_cache.tar
    elif CACHE_URL:
        !wget -q -O {WORK}/kws_cache.tar "{CACHE_URL}"
    !tar -xf {WORK}/kws_cache.tar -C data && rm -f {WORK}/kws_cache.tar
!ls -la data/kws_cache
assert os.path.exists('data/kws_cache/index.json'), 'cache missing: gdown failed (quota / sharing); use 2b to rebuild'

In [ ]:
# 2b. Rebuild the cache from the public dataset (CC BY 4.0), ~5 min on a T4.
!mkdir -p data/speech_commands_v2 data/speech_commands_test_v2
!wget -q -O {WORK}/sc.tar.gz http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!wget -q -O {WORK}/sct.tar.gz http://download.tensorflow.org/data/speech_commands_test_set_v0.02.tar.gz
!tar -xzf {WORK}/sc.tar.gz -C data/speech_commands_v2 && tar -xzf {WORK}/sct.tar.gz -C data/speech_commands_test_v2
!python audio/kws/cache.py --root data/speech_commands_v2 --test-root data/speech_commands_test_v2 --out data/kws_cache 2>&1 | grep -v '^  '
!tar -cf {WORK}/kws_cache.tar -C data kws_cache && ls -la {WORK}/kws_cache.tar  # download this once and share it for 2a

In [ ]:
# S. Results sync: after every run, JSON/MD/PNG/CSV and small checkpoints (<= 2 MB, ours are 0.1-0.5 MB) go to the
# private results repo; a new runtime gets them back and continues where the previous one died.
# Token: fine-grained PAT in the notebook secrets as RESULTS_TOKEN (Colab: key icon, notebook access ON;
# Kaggle: Add-ons -> Secrets). Repository access = Only select repositories -> bitladder-results (a token without it
# gets 404 on clone), Permissions -> Contents: read and write. Fails hard if the sync cannot start; set
# ALLOW_NO_SYNC = True only to run on purpose without it.
import datetime
from bitladder.results_sync import ResultsSync
RESULTS_REPO = 'Vzzzz/bitladder-results'
LABEL = f"{'kaggle' if ON_KAGGLE else 'colab'}-{datetime.date.today()}"
ALLOW_NO_SYNC = False

def _secret(name):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)

try:
    sync = ResultsSync(RESULTS_REPO, _secret('RESULTS_TOKEN'), LABEL, work_dir='/tmp')
    print('results sync on ->', RESULTS_REPO, '/', LABEL)
    print('restored from the repo:', sync.restore('results'), 'file(s)')
except Exception as e:
    msg = f'{type(e).__name__}: {str(e)[:300]}'
    if not ALLOW_NO_SYNC:
        raise RuntimeError('results sync could not start: ' + msg) from e
    print('results sync OFF:', msg)
    class _NoSync:
        def push(self, *a, **k):
            return 0
        def restore(self, *a, **k):
            return 0
    sync = _NoSync()

In [ ]:
# 3a. (2026-10-23) Branch bits at k=8 / k=16 on a 4-bit base, pair {stop, go}, 3 seeds. Done; kept for reference.
B = '--keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0'
run('python audio/kws/run.py --epochs 30 --name kws12_fp32_s0')
sync.push('results')
for s in (0, 1, 2):
    run(f'python audio/kws/run.py {B} --epochs 30 --seed {s} --name base8_unk1_fp32_s{s}')
    sync.push('results')
    run(f'python audio/kws/run.py {B} --epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8 '
        f'--init results/base8_unk1_fp32_s{s}.pt --seed {s} --name base8_unk1_u4a8_s{s}')
    sync.push('results')
    for k in (8, 16):
        for cfg, a in [('fp32', ''), ('u8a8', '--scheme uniform --wbits 8 --abits 8 --no-first-last'),
                       ('u4a8', '--scheme uniform --wbits 4 --abits 8 --no-first-last'),
                       ('t158a8', '--scheme ternary --abits 8 --no-first-last')]:
            run(f'python audio/kws/expand.py --base results/base8_unk1_u4a8_s{s}.json --new stop,go --epochs 20 '
                f'--k {k} --seed {s} {a} --name exp_k{k}_base-u4a8_branch-{cfg}_s{s}')
            sync.push('results')

In [ ]:
# 3b. (2026-10-24) Second pair {on, off} + depthwise-separable branch (k=16 -> 7.1k params, inside the 10k cap), 3 seeds.
# Both pairs get a fresh 4-bit base. ~60 min on a T4.
PAIRS = {'stopgo': ('yes,no,up,down,left,right,on,off', 'stop,go'),
         'onoff':  ('yes,no,up,down,left,right,stop,go', 'on,off')}
BRANCHES = [('fp32', ''), ('u4a8', '--scheme uniform --wbits 4 --abits 8 --no-first-last'),
            ('t158a8', '--scheme ternary --abits 8 --no-first-last')]
for pair, (base_kw, new) in PAIRS.items():
    B = f'--keywords {base_kw} --exclude {new} --unknown-ratio 1.0'
    for s in (0, 1, 2):
        run(f'python audio/kws/run.py {B} --epochs 30 --seed {s} --name base8_{pair}_fp32_s{s}')
        sync.push('results')
        run(f'python audio/kws/run.py {B} --epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8 '
            f'--init results/base8_{pair}_fp32_s{s}.pt --seed {s} --name base8_{pair}_u4a8_s{s}')
        sync.push('results')
        for arch, extra in [('k16ds', '--k 16 --separable'), ('k8', '--k 8'), ('k16', '--k 16')]:
            for cfg, a in BRANCHES:
                run(f'python audio/kws/expand.py --base results/base8_{pair}_u4a8_s{s}.json --new {new} --epochs 20 '
                    f'{extra} --seed {s} {a} --name exp_{pair}_{arch}_base-u4a8_branch-{cfg}_s{s}')
                sync.push('results')

In [ ]:
# 3c. (2026-10-24) Layer analysis of the 12-class model: weight / activation stats and one-shot sensitivity heatmap.
run('python audio/kws/run.py --epochs 30 --name kws12_fp32_s0')
sync.push('results')
run('python audio/kws/analyze.py --run results/kws12_fp32_s0.json --split validation --batches 10', show=('|', '#', 'wrote'))
sync.push('results')
from IPython.display import Image, display
display(Image('results/kws12_fp32_s0_sensitivity.png'))

In [ ]:
# 3d-1. Base size and time stride, pair {stop, go}, seed 0. ~45 min on a T4. Re-run after a disconnect: finished runs are skipped.
B = '--keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0'
Q = '--epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8'
BR = '--new stop,go'
for name, extra in [('b64x4', '--channels 64 --blocks 4'), ('b128x6', '--channels 128 --blocks 6'),
                    ('b64x4ts1', '--channels 64 --blocks 4 --time-stride 1')]:
    fp, q = f'base8_stopgo_{name}_fp32_s0', f'base8_stopgo_{name}_u4a8_s0'
    run_once(f'python audio/kws/run.py {B} {extra} --epochs 30 --name {fp}', fp, ckpt=True)
    sync.push('results')
    run_once(f'python audio/kws/run.py {B} {extra} {Q} --init results/{fp}.pt --name {q}', q, ckpt=True)
    sync.push('results')
    for arch, a in [('k16ds-u4a8', '--k 16 --separable --scheme uniform --wbits 4 --abits 8 --no-first-last'), ('k16-fp32', '--k 16')]:
        n = f'exp_{name}_{arch}_e20_s0'
        run_once(f'python audio/kws/expand.py --base results/{q}.json {BR} --epochs 20 {a} --name {n}', n)
        sync.push('results')

In [ ]:
# 3d-2. Branch grid on the standard base (needs base8_stopgo_b64x4_u4a8_s0 from 3d-1): epochs, lr, taps. ~15 min.
BR = '--new stop,go --k 16 --separable --scheme uniform --wbits 4 --abits 8 --no-first-last'
for ep in (20, 40):
    for lr in ('1e-3', '3e-3', '1e-2'):
        for taps in (2, 4):
            n = f'exp_grid_k16ds-u4a8_e{ep}_lr{lr}_t{taps}_s0'
            run_once(f'python audio/kws/expand.py --base results/base8_stopgo_b64x4_u4a8_s0.json {BR} --epochs {ep} --lr {lr} '
                     f'--taps {taps} --name {n}', n)
            sync.push('results')

In [ ]:
# 3e. Big base, 3 seeds, one pair per run of this cell (set PAIR). fp32 base only (QAT of the big base is 3d-1's job);
# branches: full-width k=16 separable 4-bit, and two in-budget variants (taps 2 / 3, k=8, 4-bit). ~80 min per pair on a T4.
# Re-run after a disconnect: seeds already in results/ are skipped (a base whose .pt is gone is retrained first).
PAIR = 'stopgo'   # or 'onoff'
PAIRS = {'stopgo': ('yes,no,up,down,left,right,on,off', 'stop,go'), 'onoff': ('yes,no,up,down,left,right,stop,go', 'on,off')}
base_kw, new = PAIRS[PAIR]
B = f'--keywords {base_kw} --exclude {new} --unknown-ratio 1.0 --channels 128 --blocks 6'
BR = f'--new {new} --scheme uniform --wbits 4 --abits 8 --no-first-last'
BRANCHES = [('k16ds-t4', '--k 16 --separable --taps 4'), ('k8ds-t2', '--k 8 --separable --taps 2'), ('k8ds-t3', '--k 8 --separable --taps 3')]
for s in (0, 1, 2):
    base = f'base8_{PAIR}_b128x6_fp32_s{s}'
    names = [f'exp_{PAIR}_b128x6_{arch}_u4a8_s{s}' for arch, _ in BRANCHES]
    if all(done(n) for n in names):
        print(f'skip seed {s}: all branches done'); continue
    run_once(f'python audio/kws/run.py {B} --epochs 30 --seed {s} --name {base}', base, ckpt=True)
    sync.push('results')
    for (arch, a), n in zip(BRANCHES, names):
        run_once(f'python audio/kws/expand.py --base results/{base}.json {BR} --epochs 20 --seed {s} {a} --name {n}', n)
        sync.push('results')

In [ ]:
# 3f. Methods on the standard base (needs base8_stopgo_b64x4_fp32_s0 / _u4a8_s0 and b128x6_fp32_s0 from 3d-1: their .pt
# must be in results/, i.e. 3d-1 ran in this runtime; otherwise re-run 3d-1 first, it skips what is done). ~35 min.
#   (a) mixed precision from the analyzer: profile under a 1-pt / 2-pt one-shot budget, then QAT from the fp32 checkpoint
#   (b) ternary variants for the branch: absmean (BitNet) vs TWN vs TTQ, k=8 dense and k=16 separable
#   (c) KD: 64x4 student distilled from the 128x6 teacher, then the branch
B = '--keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0'
BR = '--new stop,go --epochs 20'
run_once('python audio/kws/analyze.py --run results/base8_stopgo_b64x4_fp32_s0.json --no-heatmap --batches 4',
         'base8_stopgo_b64x4_fp32_s0_analysis', show=('|', 'wrote'))
sync.push('results')
for budget in ('0.01', '0.02'):
    n = f'base8_stopgo_b64x4_mix{budget}_a8_s0'
    run_once(f'python audio/kws/run.py {B} --epochs 15 --lr 1e-3 --scheme uniform --wbits 8 --abits 8 '
             f'--mixed-from results/base8_stopgo_b64x4_fp32_s0_analysis.json --mixed-budget {budget} '
             f'--init results/base8_stopgo_b64x4_fp32_s0.pt --name {n}', n)
    sync.push('results')
for scheme in ('ternary', 'twn', 'ttq'):
    for arch, a in [('k8', '--k 8'), ('k16ds', '--k 16 --separable')]:
        n = f'exp_b64x4_{arch}_{scheme}-a8_s0'
        run_once(f'python audio/kws/expand.py --base results/base8_stopgo_b64x4_u4a8_s0.json {BR} {a} --scheme {scheme} --abits 8 '
                 f'--no-first-last --name {n}', n)
        sync.push('results')
kd, kdq = 'base8_stopgo_b64x4kd_fp32_s0', 'base8_stopgo_b64x4kd_u4a8_s0'
run_once(f'python audio/kws/run.py {B} --channels 64 --blocks 4 --epochs 30 --teacher results/base8_stopgo_b128x6_fp32_s0.json --name {kd}', kd, ckpt=True)
sync.push('results')
run_once(f'python audio/kws/run.py {B} --channels 64 --blocks 4 --epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8 '
         f'--init results/{kd}.pt --teacher results/base8_stopgo_b128x6_fp32_s0.json --name {kdq}', kdq, ckpt=True)
sync.push('results')
run_once(f'python audio/kws/expand.py --base results/{kdq}.json {BR} --k 16 --separable --scheme uniform --wbits 4 --abits 8 '
         f'--no-first-last --name exp_b64x4kd_k16ds-u4a8_s0', 'exp_b64x4kd_k16ds-u4a8_s0')
sync.push('results')

In [ ]:
# 4. Collect a table; everything is already in the results repo (cell S), the download is a fallback.
import json, glob, os, pandas as pd
rows = []
for f in sorted(glob.glob('results/exp_*.json')):
    r = json.load(open(f))
    rows.append({'name': r['name'], 'new': '-'.join(r['new_words']), 'k': r['k'], 'separable': r.get('separable', False),
                 'seed': r['seed'], 'branch': r['config']['scheme'], 'wbits': r['config']['wbits'],
                 'branch_params': r['branch_params'], 'branch_bytes': r['branch_bytes'],
                 'frr_ungated': r['test_macro_frr_ungated'], 'frr_soft': r.get('test_macro_frr_soft'),
                 'frr_hard': r['test_macro_frr'], 'core_fires_on_new': r['core_fires_on_new']})
df = pd.DataFrame(rows)
if len(df):
    display(df.groupby(['new', 'k', 'separable', 'branch', 'wbits'])['frr_ungated'].agg(['mean', 'std', 'count']))
    df.to_csv('results/summary.csv', index=False)
    sync.push('results')
else:
    print('no exp_*.json in results/: nothing ran in this session')
!tar -czf {WORK}/results_json.tgz $(ls results/*.json results/*.md results/*.png results/*.csv 2>/dev/null) && ls -la {WORK}/results_json.tgz

In [ ]:
# 5. MNIST sandbox: bit ladder x widths, smallest model in bytes at a given accuracy. ~15 min on a T4.
run_once('python sandbox_mnist/sweep.py --epochs 10 --widths 4,6,8,12,16 --act-bits 0 8 --quiet --csv results/mnist_ladder.csv',
         'mnist_ladder.csv', show=('|', 'Pareto', 'wrote'))
sync.push('results')